# BingePlay Streaming Analytics


In [7]:
import pandas as pd
from sqlalchemy import create_engine
from IPython.display import display

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# Edit these for your machine (if the password has special characters, wrap it with urllib.parse.quote_plus)
USER, PASSWORD, HOST, DB = "root", "your_password", "localhost", "bingeplay"
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DB}")

## Setup check - row counts and the NULL trap

In [8]:
query = """
SELECT 'users' AS tbl, COUNT(*) AS row_count FROM users
UNION ALL SELECT 'subscriptions', COUNT(*) FROM subscriptions
UNION ALL SELECT 'shows', COUNT(*) FROM shows
UNION ALL SELECT 'watch_sessions', COUNT(*) FROM watch_sessions
UNION ALL SELECT 'ratings', COUNT(*) FROM ratings
UNION ALL SELECT 'watch_sessions with NULL user_id', COUNT(*) FROM watch_sessions WHERE user_id IS NULL;
"""
pd.read_sql(query, engine)

RuntimeError: 'cryptography' package is required for sha256_password or caching_sha2_password auth methods

Counts match the brief (3,000 / 4,497 / 100 / 100,351 / 5,000) and there are exactly 2 sessions with a NULL `user_id`.

# Tier 1 - Foundations

## Q1 - Active revenue
**Answer: 2,340 active subscriptions, total monthly revenue ₹7,84,260**

Active = `status = 'active'` AND (`end_date IS NULL` OR `end_date > '2024-06-30'`). Every active row has `end_date IS NULL`, so the `IS NULL` branch is what keeps them in.
*Note:* 49 of these active subscriptions have a `start_date` after 30 Jun 2024 (July starts). The brief's definition does not exclude them; a stricter "already started by 30 Jun" reading gives 2,291 subscriptions / ₹7,70,309.

In [4]:
query = """
SELECT COUNT(*)                AS active_subscriptions,
       SUM(monthly_price_inr)  AS monthly_revenue_inr
FROM subscriptions
WHERE status = 'active'
  AND (end_date IS NULL OR end_date > '2024-06-30');
"""

pd.read_sql(query, engine)

RuntimeError: 'cryptography' package is required for sha256_password or caching_sha2_password auth methods

## Q2 - Signup momentum
**Answer: May and June tie for the highest signups, 600 each.**

| Month | Signups |
|---|---|
| Jan | 350 |
| Feb | 400 |
| Mar | 500 |
| Apr | 550 |
| May | 600 |
| Jun | 600 |

Signups grew every month from January (350) and plateaued at 600 in May and June, so there is no single winner; the second query shows the tie explicitly.

In [ ]:
query = """
SELECT MONTH(signup_date) AS month,
       COUNT(*)           AS signup_count
FROM users
WHERE YEAR(signup_date) = 2024
GROUP BY MONTH(signup_date)
ORDER BY month;
"""

query_2 = """
SELECT MONTH(signup_date) AS month,
       COUNT(*)           AS signup_count
FROM users
WHERE YEAR(signup_date) = 2024
GROUP BY MONTH(signup_date)
ORDER BY signup_count DESC
LIMIT 3;
"""

display(pd.read_sql(query, engine))
display(pd.read_sql(query_2, engine))

   month  signup_count
0      1           350
1      2           400
2      3           500
3      4           550
4      5           600
5      6           600

   month  signup_count
0      6           600
1      5           600
2      4           550

## Q3 - Device analytics
**Answer** (the 2 NULL-user sessions are excluded, leaving 100,349 sessions):

| Device | Sessions | Watch minutes | Avg min/session | Completion rate |
|---|---|---|---|---|
| Mobile | 50,172 | 1,504,355 | 29.98 | 60.24% |
| TV | 27,981 | 840,595 | 30.04 | 59.98% |
| Laptop | 15,105 | 453,434 | 30.02 | 60.51% |
| Tablet | 7,091 | 210,733 | 29.72 | 59.79% |

In [ ]:
query = """
SELECT device_type,
       COUNT(*)                         AS total_sessions,
       SUM(watch_minutes)               AS total_watch_minutes,
       ROUND(AVG(watch_minutes), 2)     AS avg_minutes_per_session,
       ROUND(100 * AVG(completed), 2)   AS completion_rate_pct
FROM watch_sessions
WHERE user_id IS NOT NULL          -- brief: exclude the 2 NULL-user sessions
GROUP BY device_type
ORDER BY total_sessions DESC;
"""

pd.read_sql(query, engine)

  device_type  total_sessions  total_watch_minutes  avg_minutes_per_session  completion_rate_pct
0      Mobile           50172              1504355                    29.98                60.24
1          TV           27981               840595                    30.04                59.98
2      Laptop           15105               453434                    30.02                60.51
3      Tablet            7091               210733                    29.72                59.79

## Q4 - Rating distribution
**Answer: 71.34% of all ratings are 4 or 5 stars.**

| Stars | Ratings | Share |
|---|---|---|
| 1 | 234 | 4.68% |
| 2 | 352 | 7.04% |
| 3 | 847 | 16.94% |
| 4 | 1,781 | 35.62% |
| 5 | 1,786 | 35.72% |

Ratings are strongly positive: 4 and 5 stars are nearly equal (35.62% / 35.72%) and together make up over 70% of the 5,000 ratings.

In [ ]:
query = """
SELECT stars,
       COUNT(*) AS rating_count,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_ratings
FROM ratings
GROUP BY stars
ORDER BY stars;
"""

query_2 = """
SELECT ROUND(100.0 * SUM(CASE WHEN stars >= 4 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_4_or_5_stars
FROM ratings;
"""

display(pd.read_sql(query, engine))
display(pd.read_sql(query_2, engine))

   stars  rating_count  pct_of_ratings
0      1           234            4.68
1      2           352            7.04
2      3           847           16.94
3      4          1781           35.62
4      5          1786           35.72

   pct_4_or_5_stars
0             71.34

## Q5 - Originals vs acquired
| Group | Shows | Avg IMDb | Avg release year |
|---|---|---|---|
| BingePlay Original | 30 | 7.92 | 2020.37 |
| Acquired | 70 | 6.63 | 2020.73 |

**Answer:** Originals rate better, averaging 7.92 vs 6.63 for acquired content, a gap of 1.29 IMDb points (about 19% higher), even though the two groups have similar average release years.

In [ ]:
query = """
SELECT CASE WHEN is_original = 1 THEN 'BingePlay Original' ELSE 'Acquired' END AS content_type,
       COUNT(*)                       AS num_shows,
       ROUND(AVG(imdb_rating), 2)     AS avg_imdb_rating,
       ROUND(AVG(release_year), 2)    AS avg_release_year
FROM shows
GROUP BY is_original
ORDER BY is_original DESC;
"""

pd.read_sql(query, engine)

         content_type  num_shows  avg_imdb_rating  avg_release_year
0  BingePlay Original         30             7.92           2020.37
1            Acquired         70             6.63           2020.73

# Tier 2 - Joins & Subqueries

## Q6 - Binge day detection (Q2 2024)
**Answer: 414 binge days in total. Top user: U02956 with 8 binge days.**

A binge day is grouped on (user, show, date) with 5+ sessions, not on (user, date). The runner-up has 6 (U02292), so the top user is unambiguous.

In [ ]:
query = """
-- binge day = same user + same show + same calendar date with 5+ sessions, in Q2 2024
WITH binge_days AS (
    SELECT user_id, show_id, session_date, COUNT(*) AS sessions_that_day
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date BETWEEN '2024-04-01' AND '2024-06-30'
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(*) >= 5
),
per_user AS (
    SELECT user_id, COUNT(*) AS binge_days
    FROM binge_days
    GROUP BY user_id
)
SELECT (SELECT COUNT(*) FROM binge_days) AS total_binge_days,
       user_id                           AS top_user,
       binge_days                        AS top_user_binge_days
FROM per_user
ORDER BY binge_days DESC, user_id
LIMIT 5;
"""

pd.read_sql(query, engine)

   total_binge_days top_user  top_user_binge_days
0               414   U02956                    8
1               414   U02292                    6
2               414   U00118                    5
3               414   U02334                    5
4               414   U02169                    4

## Q7 - Q1 signups who never watched (THE NULL TRAP)
**Answer: 226 users**
**Total Q1 signups: 1,250** (18.08% never watched)

`NOT IN (SELECT user_id FROM watch_sessions)` returns **0** because the subquery contains NULLs (the third query demonstrates this). `LEFT JOIN ... IS NULL` and `NOT EXISTS` both give 226.

In [ ]:
query = """
-- Correct: LEFT JOIN ... IS NULL (immune to the NULL user_id rows in watch_sessions)
SELECT COUNT(*)                                        AS q1_signups,
       SUM(CASE WHEN ws.user_id IS NULL THEN 1 ELSE 0 END) AS never_watched
FROM users u
LEFT JOIN (SELECT DISTINCT user_id FROM watch_sessions WHERE user_id IS NOT NULL) ws
       ON ws.user_id = u.user_id
WHERE u.signup_date BETWEEN '2024-01-01' AND '2024-03-31';
"""

query_2 = """
-- Same answer via NOT EXISTS
SELECT COUNT(*) AS never_watched
FROM users u
WHERE u.signup_date BETWEEN '2024-01-01' AND '2024-03-31'
  AND NOT EXISTS (SELECT 1 FROM watch_sessions ws WHERE ws.user_id = u.user_id);
"""

query_3 = """
-- The TRAP: NOT IN against a column containing NULLs silently returns 0
SELECT COUNT(*) AS never_watched_WRONG
FROM users u
WHERE u.signup_date BETWEEN '2024-01-01' AND '2024-03-31'
  AND u.user_id NOT IN (SELECT user_id FROM watch_sessions);
"""

display(pd.read_sql(query, engine))
display(pd.read_sql(query_2, engine))
display(pd.read_sql(query_3, engine))

   q1_signups  never_watched
0        1250            226

   never_watched
0            226

   never_watched_WRONG
0                    0

## Q8 - The over-paying Premium/Family users
**Answer: 212 Premium/Family users** have no watched show requiring Premium or Family (current plan = most recent `active` subscription).

*Interpretation note:* the anti-existence (`NOT EXISTS`) pattern from the brief also counts users with **zero sessions**, who vacuously never watched a Premium show. Of the 212, 205 never watched anything at all, and only **7** actively watched and watched Basic-tier shows only. The 205 non-watchers are a different kind of downgrade/engagement target.

In [ ]:
query = """
-- current plan = plan of the user's most recent ACTIVE subscription
WITH current_sub AS (
    SELECT user_id, plan,
           ROW_NUMBER() OVER (PARTITION BY user_id
                              ORDER BY start_date DESC, subscription_id DESC) AS rn
    FROM subscriptions
    WHERE status = 'active'
),
pf_users AS (
    SELECT user_id FROM current_sub
    WHERE rn = 1 AND plan IN ('Premium', 'Family')
)
SELECT COUNT(*) AS basic_only_premium_family_users,
       SUM(CASE WHEN EXISTS (SELECT 1 FROM watch_sessions w WHERE w.user_id = p.user_id)
                THEN 1 ELSE 0 END)                       AS of_which_actually_watched
FROM pf_users p
WHERE NOT EXISTS (          -- anti-existence: no watched show needs Premium/Family
    SELECT 1
    FROM watch_sessions w
    JOIN shows sh ON sh.show_id = w.show_id
    WHERE w.user_id = p.user_id
      AND sh.min_plan IN ('Premium', 'Family')
);
"""

pd.read_sql(query, engine)

   basic_only_premium_family_users  of_which_actually_watched
0                              212                          7

## Q9 - Upgrade success cohort
**Answer: 55 users, average 64.96 days from signup to first upgrade.**

Cohort = January 2024 signups whose earliest subscription (`ROW_NUMBER() ... ORDER BY start_date`) was Basic, who later had a Premium/Family subscription, and who still hold an open `active` subscription.

In [ ]:
query = """
WITH ranked AS (
    SELECT s.user_id, s.plan, s.start_date, s.status, s.end_date,
           ROW_NUMBER() OVER (PARTITION BY s.user_id ORDER BY s.start_date, s.subscription_id) AS rn
    FROM subscriptions s
),
jan_basic_starters AS (       -- Jan-2024 signups whose very first sub was Basic
    SELECT u.user_id, u.signup_date
    FROM users u
    JOIN ranked r ON r.user_id = u.user_id AND r.rn = 1
    WHERE u.signup_date BETWEEN '2024-01-01' AND '2024-01-31'
      AND r.plan = 'Basic'
),
first_upgrade AS (            -- earliest later sub on a higher plan
    SELECT r.user_id, MIN(r.start_date) AS upgrade_date
    FROM ranked r
    WHERE r.rn > 1 AND r.plan IN ('Premium', 'Family')
    GROUP BY r.user_id
),
still_active AS (
    SELECT DISTINCT user_id
    FROM subscriptions
    WHERE status = 'active' AND (end_date IS NULL OR end_date > '2024-06-30')
)
SELECT COUNT(*)                                                    AS upgraded_and_active_users,
       ROUND(AVG(DATEDIFF(fu.upgrade_date, b.signup_date)), 2)     AS avg_days_signup_to_upgrade
FROM jan_basic_starters b
JOIN first_upgrade fu ON fu.user_id = b.user_id
JOIN still_active  sa ON sa.user_id = b.user_id;
"""

pd.read_sql(query, engine)

   upgraded_and_active_users  avg_days_signup_to_upgrade
0                         55                       64.96

## Q10 - Cliffhanger comebacks
**Answer: 4,345 comeback events. Top show: S088 - "Rayalaseema Raga" with 64 events.**

Each unique (user, show, incomplete_date) is counted once via `SELECT DISTINCT` on a self-join (same user and show, second session 1 to 7 days later). The runner-up is S005 with 61.

In [ ]:
query = """
WITH comebacks AS (           -- one row per unique (user, show, incomplete_date)
    SELECT DISTINCT a.user_id, a.show_id, a.session_date AS incomplete_date
    FROM watch_sessions a
    JOIN watch_sessions b
      ON  b.user_id = a.user_id
      AND b.show_id = a.show_id
      AND DATEDIFF(b.session_date, a.session_date) BETWEEN 1 AND 7
    WHERE a.completed = 0
      AND a.user_id IS NOT NULL
)
SELECT c.show_id, sh.title,
       COUNT(*)                         AS comeback_events,
       SUM(COUNT(*)) OVER ()            AS total_events_all_shows
FROM comebacks c
JOIN shows sh ON sh.show_id = c.show_id
GROUP BY c.show_id, sh.title
ORDER BY comeback_events DESC, c.show_id
LIMIT 5;
"""

pd.read_sql(query, engine)

  show_id             title  comeback_events  total_events_all_shows
0    S088  Rayalaseema Raga               64                    4345
1    S005   Show Special 27               61                    4345
2    S019   Founder Diaries               61                    4345
3    S024   Show Special 18               59                    4345
4    S065   Show Special 21               58                    4345

# Tier 3 - The Killer Questions

## Q11 - Consecutive-week engagement
**Answer: 1,675 users have a streak of 4+ consecutive ISO weeks. Longest streak: 26 weeks. Example user: U00213.**

7 users share the 26-week streak (every ISO week of Jan-Jun 2024): U00213, U00529, U01259, U01658, U01793, U01985, U02770. Method: distinct (user, week), subtract `ROW_NUMBER()` from the week number so each unbroken run shares one key, then `GROUP BY (user, key)` and count.

In [ ]:
query = """
-- Gaps-and-islands. All sessions fall in 2024 (ISO weeks 1-26), so the ISO week number is a safe ordinal.
WITH weeks AS (
    SELECT DISTINCT user_id, WEEK(session_date, 3) AS iso_week
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),
numbered AS (
    SELECT user_id, iso_week,
           iso_week - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY iso_week) AS island_key
    FROM weeks
),
streaks AS (
    SELECT user_id, island_key, COUNT(*) AS streak_weeks,
           MIN(iso_week) AS first_week, MAX(iso_week) AS last_week
    FROM numbered
    GROUP BY user_id, island_key
),
best AS (
    SELECT user_id, MAX(streak_weeks) AS best_streak
    FROM streaks
    GROUP BY user_id
)
SELECT (SELECT COUNT(*) FROM best WHERE best_streak >= 4)  AS users_with_4plus_week_streak,
       (SELECT MAX(best_streak) FROM best)                 AS longest_streak_weeks,
       (SELECT COUNT(*) FROM best
         WHERE best_streak = (SELECT MAX(best_streak) FROM best)) AS users_with_longest_streak,
       (SELECT MIN(user_id) FROM best
         WHERE best_streak = (SELECT MAX(best_streak) FROM best)) AS example_user;
"""

query_2 = """
WITH weeks AS (SELECT DISTINCT user_id, WEEK(session_date, 3) AS iso_week
               FROM watch_sessions WHERE user_id IS NOT NULL),
numbered AS (SELECT user_id, iso_week,
                    iso_week - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY iso_week) AS island_key
             FROM weeks)
SELECT user_id, COUNT(*) AS streak_weeks, MIN(iso_week) AS first_week, MAX(iso_week) AS last_week
FROM numbered
GROUP BY user_id, island_key
ORDER BY streak_weeks DESC, user_id
LIMIT 5;
"""

display(pd.read_sql(query, engine))
display(pd.read_sql(query_2, engine))

   users_with_4plus_week_streak  longest_streak_weeks  users_with_longest_streak example_user
0                          1675                    26                          7       U00213

  user_id  streak_weeks  first_week  last_week
0  U00213            26           1         26
1  U00529            26           1         26
2  U01259            26           1         26
3  U01658            26           1         26
4  U01793            26           1         26

## Q12 - Churn signal detection
**Answer: 521 users** dropped 50% or more in June vs May (May-active users only).

97 of them watched nothing at all in June (100% drop) and 424 still watched a little. The query uses conditional aggregation inside a CTE rather than `LAG`, so users with no June rows are not silently dropped; counting only users with some June activity would give 424. The first rows are shown below (largest drops first).

In [ ]:
query = """
-- Conditional aggregation (not LAG): a user who watched in May but NOT at all in June must count as a 100% drop,
-- and LAG over only the months a user has rows would silently skip them.
WITH monthly AS (
    SELECT user_id,
           SUM(CASE WHEN MONTH(session_date) = 5 THEN watch_minutes ELSE 0 END) AS may_minutes,
           SUM(CASE WHEN MONTH(session_date) = 6 THEN watch_minutes ELSE 0 END) AS june_minutes
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date BETWEEN '2024-05-01' AND '2024-06-30'
    GROUP BY user_id
),
drops AS (
    SELECT user_id, may_minutes, june_minutes,
           ROUND(100.0 * (may_minutes - june_minutes) / may_minutes, 2) AS drop_pct
    FROM monthly
    WHERE may_minutes > 0
)
SELECT d.user_id, u.name, d.may_minutes, d.june_minutes, d.drop_pct
FROM drops d
JOIN users u ON u.user_id = d.user_id
WHERE d.drop_pct >= 50
ORDER BY d.drop_pct DESC, d.may_minutes DESC, d.user_id;
"""

pd.read_sql(query, engine)

    user_id                 name  may_minutes  june_minutes  drop_pct
0    U01503           Myra Patil          759             0    100.00
1    U01397        Shreya Shenoy          727             0    100.00
2    U00292       Kavitha Kamath          594             0    100.00
3    U02199          Advait Shah          559             0    100.00
4    U00883         Atharv Verma          547             0    100.00
5    U02351            Karan Sen          532             0    100.00
6    U01249           Sara Kumar          494             0    100.00
7    U00804          Kiara Ghosh          475             0    100.00
8    U01423         Yuvraj Patil          460             0    100.00
9    U02683           Krish Jain          424             0    100.00
10   U01353      Rohit Chowdhury          401             0    100.00
11   U00558            Radha Das          400             0    100.00
12   U00211           Ravi Menon          336             0    100.00
13   U00777         